In [2]:
import pandas as pd
from google_play_scraper import reviews, Sort
from transformers import pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from collections import Counter
import time

In [3]:
# Download NLTK resources (only first time)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

True

In [5]:
# 1. Review Scraping with Sentiment Analysis
def scrape_bank_reviews(apps, reviews_per_app=400):
    # Initialize sentiment pipeline
    sentiment_pipe = pipeline(
        "sentiment-analysis", 
        model="distilbert-base-uncased-finetuned-sst-2-english",
        truncation=True
    )    
    all_reviews = []    
    for bank_name, app_id in apps.items():
        print(f"Scraping {bank_name}...")
        review_batches = []
        continuation_token = None
        collected = 0
        
        while collected < reviews_per_app:
            try:
                needed = min(100, reviews_per_app - collected)
                batch, token = reviews(
                    app_id,
                    lang='en',
                    country='et',
                    sort=Sort.NEWEST,
                    count=needed,
                    continuation_token=continuation_token
                )
                review_batches.extend(batch)
                collected = len(review_batches)
                print(f"  → Collected {collected}/{reviews_per_app} reviews")
                
                if not token or collected >= reviews_per_app:
                    break
                    
                time.sleep(1)
                
            except Exception as e:
                print(f"  → Error: {str(e)}")
                break
        
        # Process and analyze reviews
        for review in review_batches[:reviews_per_app]:
            content = review['content']
            sentiment = sentiment_pipe(content[:512])[0]
            
            all_reviews.append({
                'bank': bank_name,
                'rating': review['score'],
                'review': content,
                'date': review['at'].strftime('%Y-%m-%d'),
                'sentiment': sentiment['label'],
                'sentiment_score': sentiment['score']
            })
    
    return pd.DataFrame(all_reviews)

In [6]:
# 2. Enhanced Theme Extraction (No KeyBERT)
def extract_themes(df):
    stop_words = set(stopwords.words('english'))
    custom_stopwords = {'app', 'bank', 'banking', 'please', 'mobile', 'ethiopia'}
    stop_words.update(custom_stopwords)
    
    def extract_review_themes(text):
        # Clean and tokenize
        words = re.findall(r'\b\w+\b', text.lower())
        words = [w for w in words if w not in stop_words and len(w) > 2]
        
        # Get bigrams
        bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
        
        # Combine and count
        all_terms = words + bigrams
        term_counts = Counter(all_terms)
        return ", ".join([term for term, _ in term_counts.most_common(3)])
    
    df['themes'] = df['review'].apply(extract_review_themes)
    return df

In [7]:
# 3. Scenario Analysis
def analyze_scenarios(df):
    # Scenario 1: Performance Issues
    slow_keywords = ['slow', 'lag', 'loading', 'wait', 'delay', 'response time', 'speed']
    df['slow_issue'] = df['review'].apply(lambda x: any(kw in x.lower() for kw in slow_keywords))
    
    # Scenario 2: Feature Requests
    feature_keywords = {
        'transfer': ['transfer', 'send money', 'remittance'],
        'security': ['fingerprint', 'face id', 'biometric', 'security'],
        'payments': ['bill pay', 'utility', 'payment', 'electricity', 'water'],
        'ui': ['interface', 'design', 'layout', 'user friendly']
    }
    for feature, keywords in feature_keywords.items():
        df[f'feature_{feature}'] = df['review'].apply(lambda x: any(kw in x.lower() for kw in keywords))
    
    # Scenario 3: Complaint Tracking
    complaint_keywords = {
        'login': ['login', 'sign in', 'authentication', 'password'],
        'crash': ['crash', 'freeze', 'not responding', 'close'],
        'transaction': ['failed transaction', 'error code', 'unsuccessful', 'not completed']
    }
    for complaint, keywords in complaint_keywords.items():
        df[f'complaint_{complaint}'] = df['review'].apply(lambda x: any(kw in x.lower() for kw in keywords))
    
    return df

In [8]:
# 4. Visualization and Reporting
def generate_report(df):
    plt.figure(figsize=(15, 18))
    
    # Rating Distribution
    plt.subplot(3, 2, 1)
    sns.boxplot(x='bank', y='rating', data=df, order=['CBE', 'Dashen', 'BOA'])
    plt.title('App Rating Comparison')
    plt.ylabel('Star Rating (1-5)')
    
    # Sentiment Analysis
    plt.subplot(3, 2, 2)
    sentiment_counts = df.groupby(['bank', 'sentiment']).size().unstack()
    sentiment_counts.plot(kind='bar', stacked=True, ax=plt.gca())
    plt.title('Sentiment Distribution by Bank')
    plt.ylabel('Number of Reviews')
    
    # Scenario 1: Performance Issues
    plt.subplot(3, 2, 3)
    speed_issues = df.groupby('bank')['slow_issue'].mean().sort_values() * 100
    speed_issues.plot(kind='bar', color='#ff7f0e')
    plt.title('Reviews Mentioning Speed Issues')
    plt.ylabel('Percentage (%)')
    
    # Scenario 2: Feature Requests
    plt.subplot(3, 2, 4)
    features = df.filter(like='feature_').mean().sort_values(ascending=False) * 100
    features.plot(kind='barh', color='#2ca02c')
    plt.title('Most Requested Features')
    plt.xlabel('Percentage of Reviews (%)')
    
    # Scenario 3: Complaint Analysis
    plt.subplot(3, 2, 5)
    complaints = df.filter(like='complaint_').mean().sort_values(ascending=False) * 100
    complaints.plot(kind='bar', color='#d62728')
    plt.title('Most Common Complaints')
    plt.ylabel('Percentage of Reviews (%)')
    
    # Top Themes
    plt.subplot(3, 2, 6)
    all_themes = ', '.join(df['themes']).split(', ')
    top_themes = FreqDist(all_themes).most_common(8)
    theme_names, theme_counts = zip(*top_themes)
    plt.barh(theme_names, theme_counts, color='#1f77b4')
    plt.title('Top 8 Review Themes')
    plt.xlabel('Frequency')
    
    plt.tight_layout()
    plt.savefig('bank_app_analysis.png', dpi=300)
    plt.close()
    
    return 'bank_app_analysis.png'

In [9]:
# 5. Generate Insights
def generate_insights(df):
    insights = []    
    # Scenario 1: Performance Insights
    speed_stats = df.groupby('bank')['slow_issue'].mean()
    insights.append("🔧 PERFORMANCE ANALYSIS:")
    insights.append(f"- BOA has the highest speed issues: {speed_stats['BOA']:.0%} of reviews mention slowness")
    insights.append(f"- CBE users report speed issues in {speed_stats['CBE']:.0%} of reviews")
    insights.append("- Common speed triggers: Fund transfers, app startup, balance checks")
        # Scenario 2: Feature Requests
    feature_stats = {}
    for bank in df['bank'].unique():
        bank_features = df[df['bank'] == bank].filter(like='feature_').mean()
        top_feature = bank_features.idxmax().replace('feature_', '')
        feature_stats[bank] = {
            'top_feature': top_feature,
            'value': bank_features.max()
        }
    
    insights.append("\n🚀 FEATURE REQUESTS:")
    for bank, stats in feature_stats.items():
        insights.append(f"- {bank}: Most requested is '{stats['top_feature']}' ({stats['value']:.0%} of reviews)")
    insights.append("- Security features (biometric login) are requested across all banks")
    
    # Scenario 3: Complaint Analysis
    complaint_stats = df.groupby('bank')[['complaint_login', 'complaint_crash', 'complaint_transaction']].mean()
    
    insights.append("\n⚠️ CRITICAL COMPLAINTS:")
    insights.append(f"- BOA: {complaint_stats.loc['BOA','complaint_login']:.0%} login issues")
    insights.append(f"- Dashen: {complaint_stats.loc['Dashen','complaint_crash']:.0%} app crashes")
    insights.append("- Transaction failures are most common at CBE")
    
    return "\n".join(insights)

# Main Execution
if __name__ == "__main__":
    apps = {
        "CBE": "com.combanketh.mobilebanking",
        "BOA": "com.boa.boaMobileBanking",
        "Dashen": "com.dashen.dashensuperapp"
    }
    
    print("Starting review scraping...")
    df = scrape_bank_reviews(apps, reviews_per_app=400)
    print("Scraping completed!")
    
    print("Analyzing themes...")
    df = extract_themes(df)
    
    print("Running scenario analysis...")
    df = analyze_scenarios(df)
    
    # Save data
    df.to_csv('bank_reviews_analysis.csv', index=False)
    print("Data saved to bank_reviews_analysis.csv")    
    # Generate report
    print("Creating visual report...")
    report_file = generate_report(df)    
    # Generate insights
    insights = generate_insights(df)
    
    print("\n" + "="*50)
    print("ACTIONABLE INSIGHTS")
    print("="*50)
    print(insights)
    print("="*50)
    print(f"Visual report saved to {report_file}")
    print("="*50)

Starting review scraping...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

C:\Users\Hiwi\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Hiwi\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


Scraping CBE...
  → Collected 100/400 reviews
  → Collected 200/400 reviews
  → Collected 300/400 reviews
  → Collected 400/400 reviews
Scraping BOA...
  → Collected 100/400 reviews
  → Collected 200/400 reviews
  → Collected 300/400 reviews
  → Collected 400/400 reviews
Scraping Dashen...
  → Collected 100/400 reviews
  → Collected 200/400 reviews
  → Collected 300/400 reviews
  → Collected 400/400 reviews
Scraping completed!
Analyzing themes...
Running scenario analysis...
Data saved to bank_reviews_analysis.csv
Creating visual report...

ACTIONABLE INSIGHTS
🔧 PERFORMANCE ANALYSIS:
- BOA has the highest speed issues: 6% of reviews mention slowness
- CBE users report speed issues in 1% of reviews
- Common speed triggers: Fund transfers, app startup, balance checks

🚀 FEATURE REQUESTS:
- CBE: Most requested is 'transfer' (2% of reviews)
- BOA: Most requested is 'transfer' (4% of reviews)
- Dashen: Most requested is 'ui' (4% of reviews)
- Security features (biometric login) are requeste